# Ethical Python Security Basics

This notebook is a beginner-friendly introduction to **defensive security** in Python. It demonstrates safe practices using local examples and synthetic data: file-integrity checks, password hygiene, safe database queries, input validation, and message authentication.

> Only assess systems and data you own or have explicit permission to test. This notebook intentionally does not include instructions for breaking into systems, stealing credentials, evading detection, or exploiting real targets.

## 1. Check a file's integrity with SHA-256
A cryptographic hash acts like a file fingerprint. Comparing a trusted hash with a newly computed hash can reveal whether a file changed. A hash is **not encryption** and cannot be used to recover the original file.

In [ ]:
import hashlib
from pathlib import Path

def sha256_file(path: str | Path) -> str:
    """Return the SHA-256 hex digest of a file, reading it in chunks."""
    digest = hashlib.sha256()
    with Path(path).open('rb') as file:
        for chunk in iter(lambda: file.read(65536), b''):
            digest.update(chunk)
    return digest.hexdigest()

# Example using in-memory bytes, so this demo does not need a local file.
sample = b'example training file\n'
print(hashlib.sha256(sample).hexdigest())

## 2. Generate strong random tokens
Use Python's `secrets` module—not `random`—for security-sensitive random values. The example creates a token for a demo; store real secrets in a secret manager or environment configuration, never commit them to a repository.

In [ ]:
import secrets

demo_token = secrets.token_urlsafe(24)
print('Example token:', demo_token)
# Do not reuse this printed demo token for a real account or service.

## 3. Check password policy without storing passwords
This illustrative policy check reports basic length and character-category properties. It does not transmit or save the input. For production authentication, use a well-reviewed password-strength library and a password-hashing algorithm such as Argon2id, scrypt, or bcrypt; never store plaintext passwords or fast hashes such as SHA-256 as password hashes.

In [ ]:
def password_policy_feedback(password: str) -> list[str]:
    """Return simple suggestions; this is not a full strength estimator."""
    suggestions = []
    if len(password) < 14:
        suggestions.append('Use at least 14 characters; a longer passphrase is better.')
    if password.lower() == password or password.upper() == password:
        suggestions.append('Consider mixing character types, or use a long unique passphrase.')
    if password.isalpha() or password.isdigit():
        suggestions.append('Avoid passwords made only of letters or only of digits.')
    if not suggestions:
        suggestions.append('Basic checks passed; use a unique password and a password manager.')
    return suggestions

# Use only a made-up example here. Never paste a real password into a notebook.
password_policy_feedback('correct-horse-battery-example')

## 4. Prevent SQL injection with parameterized queries
Never build SQL by concatenating user-provided strings. Bind values as parameters so the database treats them as data. This example uses a temporary in-memory SQLite database and synthetic records.

In [ ]:
import sqlite3

connection = sqlite3.connect(':memory:')
connection.execute('CREATE TABLE accounts (username TEXT, role TEXT)')
connection.executemany(
    'INSERT INTO accounts (username, role) VALUES (?, ?)',
    [('alice_demo', 'reader'), ('bob_demo', 'editor')],
)

requested_username = 'alice_demo'  # Treat this as untrusted input.
row = connection.execute(
    'SELECT username, role FROM accounts WHERE username = ?',
    (requested_username,),
).fetchone()
print(row)
connection.close()

## 5. Validate input with an allowlist
Validation should define what is allowed, rather than trying to enumerate every dangerous input. Keep validation close to the boundary where data enters your program, and still use safe APIs downstream.

In [ ]:
import re

USERNAME_PATTERN = re.compile(r'^[a-zA-Z0-9_]{3,30}$')

def is_valid_username(value: str) -> bool:
    """Accept 3-30 ASCII letters, digits, or underscores."""
    return bool(USERNAME_PATTERN.fullmatch(value))

print(is_valid_username('learner_01'))  # True
print(is_valid_username('../secret'))  # False

## 6. Keep file access inside an intended directory
When an application accepts a filename, resolve it and verify that it remains beneath the allowed directory. This helps prevent path traversal bugs. Use a temporary directory for the demonstration.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

def safe_child_path(root: Path, user_filename: str) -> Path:
    """Resolve a requested path and reject anything outside root."""
    root = root.resolve()
    candidate = (root / user_filename).resolve()
    if candidate != root and root not in candidate.parents:
        raise ValueError('Requested path is outside the permitted directory')
    return candidate

with TemporaryDirectory() as temporary_directory:
    allowed_root = Path(temporary_directory)
    print(safe_child_path(allowed_root, 'notes.txt'))
    try:
        safe_child_path(allowed_root, '../outside.txt')
    except ValueError as error:
        print(type(error).__name__ + ':', error)

## 7. Compare secret values safely
For comparing authentication tokens or MAC values, `hmac.compare_digest` avoids content-dependent early exits. It does not make an insecure authentication design secure by itself; use established protocols and libraries in production.

In [ ]:
import hmac

expected_demo_token = 'sample-token-for-this-demo'
provided_demo_token = 'sample-token-for-this-demo'
is_valid = hmac.compare_digest(expected_demo_token, provided_demo_token)
print('Token matches:', is_valid)

## 8. Authenticate a message with HMAC
An HMAC lets parties sharing a secret key check that a message has not been altered and was produced by someone with that key. The key must remain secret. Do not use this toy example as a substitute for a complete signed-request protocol.

In [ ]:
import hashlib
import hmac

demo_key = b'not-a-real-secret-key'  # Replace only in a controlled learning exercise.
message = b'example event: user signed in'
tag = hmac.new(demo_key, message, hashlib.sha256).hexdigest()

def verify_message(key: bytes, body: bytes, supplied_tag: str) -> bool:
    expected_tag = hmac.new(key, body, hashlib.sha256).hexdigest()
    return hmac.compare_digest(expected_tag, supplied_tag)

print('Message authentic:', verify_message(demo_key, message, tag))

## Further safe practice
- Learn networking and web security in deliberately vulnerable local labs such as OWASP Juice Shop or WebGoat; never target public systems without written authorization.
- Keep dependencies updated, review permissions, use multi-factor authentication, and avoid committing credentials.
- For real applications, prefer maintained security libraries and have your design reviewed.
- Before running a security test, document its scope, authorization, allowed methods, and stop conditions.